# Adaptive Deep Neural Network Inference Using Confidence-Based Early Exits

ICT-4442 Deep Learning Project, Team 15: Aditya Goyal, Raunak Panshikar, Arth Mishra, Kartikey Goyal.

Early-exit networks attach classifiers at several depths of a backbone. At inference an image stops at the first
exit whose prediction is confident enough, so easy images use less computation. This notebook contains the whole
project, end to end, on CIFAR-100:

1. **Data**: stratified train/val split, GPU augmentation, six CIFAR-100-C corruptions at five severities.
2. **Models**: ResNet-18 (Model 1), early-exit ResNet-18 used by the confidence rule (Model 2) and the
   calibrated / entropy rule (Model 3), and an early-exit ViT-Tiny (Model 4).
3. **Training** with self-distillation from the final exit into the early exits.
4. **Exit rules**: max softmax probability, entropy, per-exit temperature scaling, thresholds chosen on validation.
5. **Evaluation**: accuracy vs FLOPs, calibration, real batch-1 latency (GPU and CPU), robustness under corruption.
6. **Results** of our full-scale runs (3 seeds, ablations, follow-up study), loaded from the repository.

Code repository: https://github.com/goyaladitya05/adaptive-dnn-inference

**Runtime**: Colab with a GPU (Runtime > Change runtime type > T4 GPU). `MODE = "fast"` trains for 30 epochs
(about 1 hour for the three models); `MODE = "full"` uses the 200-epoch settings of the report (about 3.5 hours,
longer than a free Colab session may last, so mount Google Drive: finished runs are skipped when you re-run).

## 0. Setup

In [ ]:
import os, sys, json, glob, shutil, subprocess, time, math, copy, re, itertools, platform, argparse
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

MODE = os.environ.get("NB_MODE", "fast")      # "fast", "full", or "smoke" (CPU check, a few minutes)
USE_DRIVE = False                             # True: keep runs in Google Drive so they survive disconnects
SEEDS = [0]                                   # the report uses [0, 1, 2]

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/adaptive-dnn-inference"
else:
    WORK = os.path.abspath(os.environ.get("NB_WORK", "work"))
DATA = os.path.join(WORK, "data")
RUNS_DIR = os.path.join(WORK, "runs")             # runs/final-colab/<model>_s<seed>/, as the analysis expects
RESULTS_DIR = os.path.join(WORK, "results")
for d in (DATA, RUNS_DIR, os.path.join(RESULTS_DIR, "figures")):
    os.makedirs(d, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() and MODE != "smoke" else "cpu"
print("torch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


def cli(fn, *args):
    """Call one of the project's command-line entry points with the given arguments."""
    sys.argv = [fn.__name__, *map(str, args)]
    return fn()

## 1. Data pipeline

CIFAR-100 is downloaded with torchvision. 5,000 training images (50 per class, seed 0) form a validation set used
only for fitting temperatures and choosing thresholds; the official 10,000 test images are used for reporting. The
whole uint8 dataset is kept on the GPU and augmented there (random crop with reflect padding, horizontal flip;
RandAugment for the ViT), which avoids the CPU data-loader bottleneck.

In [ ]:
import argparse

import numpy as np
import torch
import torch.nn.functional as F
from torchvision.datasets import CIFAR100
from torchvision.transforms import v2

NUM_CLASSES = 100
MEAN = (0.5071, 0.4865, 0.4409)
STD = (0.2673, 0.2564, 0.2762)


def load_cifar100(root="data", val_size=5000, seed=0):
    """Returns uint8 NHWC arrays for a stratified train/val split and the official test set."""
    tr = CIFAR100(root, train=True, download=True)
    te = CIFAR100(root, train=False, download=True)
    x, y = tr.data, np.asarray(tr.targets)
    rng = np.random.default_rng(seed)
    per_class = val_size // NUM_CLASSES
    val_idx = np.concatenate([rng.permutation(np.flatnonzero(y == c))[:per_class] for c in range(NUM_CLASSES)])
    mask = np.ones(len(y), dtype=bool)
    mask[val_idx] = False
    return {
        "train": (x[mask], y[mask]),
        "val": (x[val_idx], y[val_idx]),
        "test": (te.data, np.asarray(te.targets)),
    }


def to_device(split, device):
    x, y = split
    return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous().to(device), torch.from_numpy(y).long().to(device)


def normalize(x):
    mean = torch.tensor(MEAN, device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=x.device).view(1, 3, 1, 1)
    return (x - mean) / std


def augment(x, pad=4):
    """Random crop with reflect padding and horizontal flip, done on the GPU."""
    n, _, h, w = x.shape
    x = F.pad(x, (pad, pad, pad, pad), mode="reflect")
    i = torch.randint(0, 2 * pad + 1, (n, 1), device=x.device) + torch.arange(h, device=x.device)
    j = torch.randint(0, 2 * pad + 1, (n, 1), device=x.device) + torch.arange(w, device=x.device)
    x = x[torch.arange(n, device=x.device)[:, None, None, None], torch.arange(3, device=x.device)[None, :, None, None],
          i[:, None, :, None], j[:, None, None, :]]
    flip = torch.rand(n, device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x)


class ChunkRandAugment:
    """RandAugment on the GPU; each chunk of the batch draws its own operations."""

    def __init__(self, chunk=32, num_ops=2, magnitude=9):
        self.op = v2.RandAugment(num_ops=num_ops, magnitude=magnitude)
        self.chunk = chunk

    def __call__(self, x):
        u8 = (x * 255).round().to(torch.uint8)
        return torch.cat([self.op(c) for c in u8.split(self.chunk)]).float().div_(255)


class GPULoader:
    """Keeps the whole uint8 split on the device and yields normalized float batches."""

    def __init__(self, x, y, batch_size, train=False, transform=None):
        self.x, self.y, self.bs, self.train, self.transform = x, y, batch_size, train, transform

    def __len__(self):
        return (len(self.y) + self.bs - 1) // self.bs

    def __iter__(self):
        n = len(self.y)
        idx = torch.randperm(n, device=self.y.device) if self.train else torch.arange(n, device=self.y.device)
        for k in range(0, n, self.bs):
            b = idx[k:k + self.bs]
            xb = self.x[b].float().div_(255)
            if self.train:
                xb = augment(xb)
            if self.transform is not None:
                xb = self.transform(xb)
            yield normalize(xb), self.y[b]


### Corruptions

Six corruptions of the CIFAR-100-C benchmark (Hendrycks and Dietterich, 2019), re-implemented on the GPU with the
original severity constants and re-quantised to 8 bits.

In [ ]:
"""GPU re-implementation of a subset of the CIFAR-10/100-C corruptions (Hendrycks and Dietterich, 2019).

Inputs are float NCHW tensors in [0, 1]. Severity constants follow the CIFAR-C generation code.
"""
import torch
import torch.nn.functional as F


def gaussian_noise(x, s, g):
    c = [0.04, 0.06, 0.08, 0.09, 0.10][s - 1]
    return (x + torch.randn(x.shape, generator=g, device=x.device) * c).clamp(0, 1)


def shot_noise(x, s, g):
    c = [500, 250, 100, 75, 50][s - 1]
    return (torch.poisson(x * c, generator=g) / c).clamp(0, 1)


def impulse_noise(x, s, g):
    c = [0.01, 0.02, 0.03, 0.05, 0.07][s - 1]
    u = torch.rand(x.shape, generator=g, device=x.device)
    x = x.masked_fill(u < c / 2, 0.0)
    return x.masked_fill((u >= c / 2) & (u < c), 1.0)


def _filter(x, k, mode):
    kh, kw = k.shape
    x = F.pad(x, (kw // 2, kw // 2, kh // 2, kh // 2), mode=mode)
    return F.conv2d(x, k.expand(x.shape[1], 1, kh, kw), groups=x.shape[1])


def _gauss1d(sigma, radius, device):
    t = torch.arange(-radius, radius + 1, device=device, dtype=torch.float32)
    k = torch.exp(-t ** 2 / (2 * sigma ** 2))
    return k / k.sum()


def gaussian_blur(x, s, g):
    c = [0.4, 0.6, 0.7, 0.8, 1.0][s - 1]
    k = _gauss1d(c, int(4 * c + 0.5), x.device)
    return _filter(x, torch.outer(k, k), "replicate").clamp(0, 1)


def defocus_blur(x, s, g):
    radius, alias = [(0.3, 0.4), (0.4, 0.5), (0.5, 0.6), (1.0, 0.2), (1.5, 0.1)][s - 1]
    t = torch.arange(-8, 9, device=x.device, dtype=torch.float32)
    disk = ((t[:, None] ** 2 + t[None, :] ** 2) <= radius ** 2).float()
    disk = disk / disk.sum()
    g3 = _gauss1d(alias, 1, x.device)
    disk = _filter(disk[None, None], torch.outer(g3, g3), "reflect")[0, 0]
    return _filter(x, disk, "reflect").clamp(0, 1)


def contrast(x, s, g):
    c = [0.75, 0.5, 0.4, 0.3, 0.15][s - 1]
    m = x.mean(dim=(2, 3), keepdim=True)
    return ((x - m) * c + m).clamp(0, 1)


CORRUPTIONS = {
    "gaussian_noise": gaussian_noise,
    "shot_noise": shot_noise,
    "impulse_noise": impulse_noise,
    "gaussian_blur": gaussian_blur,
    "defocus_blur": defocus_blur,
    "contrast": contrast,
}


def corrupt(name, severity, generator):
    """Returns a transform that applies the corruption and re-quantizes to 8 bits like the stored CIFAR-C."""
    fn = CORRUPTIONS[name]
    return lambda x: torch.round(fn(x, severity, generator) * 255) / 255


In [ ]:
data = load_cifar100(DATA)
if MODE == "smoke":   # tiny splits so the whole notebook can be checked on a CPU
    data = {"train": tuple(a[:512] for a in data["train"]), "val": tuple(a[:500] for a in data["val"]),
            "test": tuple(a[:500] for a in data["test"])}
    _full = load_cifar100
    load_cifar100 = lambda root="data", val_size=5000, seed=0: data  # noqa: E731
for name, (x, y) in data.items():
    print(f"{name:5s} {len(y):6d} images, per-class min/max {np.bincount(y, minlength=100).min()}/{np.bincount(y, minlength=100).max()}")

from torchvision.datasets import CIFAR100
classes = CIFAR100(DATA, train=False, download=False).classes
x, y = data["test"]
fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, i in zip(axes.flat, np.random.default_rng(3).choice(len(y), 16, replace=False)):
    ax.imshow(x[i]); ax.set_title(classes[y[i]].replace("_", " "), fontsize=7); ax.axis("off")
plt.show()

img = torch.from_numpy(x[:1]).permute(0, 3, 1, 2).float() / 255
fig, axes = plt.subplots(3, len(CORRUPTIONS) + 1, figsize=(10, 4.5))
for r, s in enumerate((1, 3, 5)):
    axes[r, 0].imshow(img[0].permute(1, 2, 0)); axes[r, 0].set_ylabel(f"severity {s}")
    for c, name in enumerate(CORRUPTIONS, 1):
        axes[r, c].imshow(corrupt(name, s, torch.Generator().manual_seed(s))(img)[0].permute(1, 2, 0))
        if r == 0: axes[r, c].set_title(name.replace("_", " "), fontsize=8)
    if r == 0: axes[r, 0].set_title("clean", fontsize=8)
for ax in axes.flat: ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 2. Models

Every model is a backbone split into segments with a classifier head after each segment (a plain network is one
segment and one head). EE-ResNet-18 has exits after its four stages; the heads of stages 1 to 3 use strided 3x3
convolutions down to 4x4 before pooling. EE-ViT-Tiny (patch 4, width 192, 12 blocks) has exits after blocks 3, 6,
9 and 12. FLOPs are counted with PyTorch's FlopCounterMode, including every head an image passes.

In [ ]:
import torch.nn as nn


class MultiExitNet(nn.Module):
    """Backbone split into segments with one classifier head after each segment.

    A plain network is the special case with a single segment and a single head.
    """

    def __init__(self, segments, heads):
        super().__init__()
        assert len(segments) == len(heads)
        self.segments = nn.ModuleList(segments)
        self.heads = nn.ModuleList(heads)

    @property
    def num_exits(self):
        return len(self.heads)

    def forward(self, x):
        outs = []
        for seg, head in zip(self.segments, self.heads):
            x = seg(x)
            outs.append(head(x))
        return outs


import torch
import torch.nn as nn
import torch.nn.functional as F



class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class PoolHead(nn.Module):
    def __init__(self, cin, num_classes):
        super().__init__()
        self.fc = nn.Linear(cin, num_classes)

    def forward(self, x):
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


class ConvHead(nn.Module):
    """Strided 3x3 convs down to 4x4 resolution, then a pooled linear classifier."""

    def __init__(self, cin, size, num_classes, width=128):
        super().__init__()
        layers = []
        while size > 4:
            layers += [nn.Conv2d(cin, width, 3, 2, 1, bias=False), nn.BatchNorm2d(width), nn.ReLU(inplace=True)]
            cin, size = width, size // 2
        self.features = nn.Sequential(*layers)
        self.pool = PoolHead(cin, num_classes)

    def forward(self, x):
        return self.pool(self.features(x))


def _stem():
    return nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))


def _stages():
    def stage(cin, cout, stride):
        return nn.Sequential(BasicBlock(cin, cout, stride), BasicBlock(cout, cout, 1))

    return [stage(64, 64, 1), stage(64, 128, 2), stage(128, 256, 2), stage(256, 512, 2)]


def resnet18(num_classes=100):
    """CIFAR ResNet-18: 3x3 stem, no max-pool, single exit."""
    return MultiExitNet([nn.Sequential(_stem(), *_stages())], [PoolHead(512, num_classes)])


def ee_resnet18(num_classes=100):
    """ResNet-18 with exits after each of the four residual stages."""
    s = _stages()
    segments = [nn.Sequential(_stem(), s[0]), s[1], s[2], s[3]]
    heads = [
        ConvHead(64, 32, num_classes),
        ConvHead(128, 16, num_classes),
        ConvHead(256, 8, num_classes),
        PoolHead(512, num_classes),
    ]
    return MultiExitNet(segments, heads)


def ee_resnet18_pool(num_classes=100):
    """Ablation: exits after each stage with only pooling and a linear layer (SDN-style cheap heads)."""
    s = _stages()
    segments = [nn.Sequential(_stem(), s[0]), s[1], s[2], s[3]]
    return MultiExitNet(segments, [PoolHead(c, num_classes) for c in (64, 128, 256, 512)])


import torch
import torch.nn as nn
import torch.nn.functional as F



class Attention(nn.Module):
    manual = False  # explicit matmuls so FlopCounterMode sees the attention core

    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        b, n, d = x.shape
        q, k, v = self.qkv(x).view(b, n, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        if self.manual:
            x = (q @ k.transpose(-2, -1) * q.shape[-1] ** -0.5).softmax(-1) @ v
        else:
            x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(b, n, d))


class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * mlp_ratio), nn.GELU(), nn.Linear(dim * mlp_ratio, dim))
        self.drop_path = drop_path

    def _drop(self, x):
        if not self.training or self.drop_path == 0:
            return x
        keep = (torch.rand(x.shape[0], 1, 1, device=x.device) >= self.drop_path).to(x.dtype)
        return x * keep / (1 - self.drop_path)

    def forward(self, x):
        x = x + self._drop(self.attn(self.norm1(x)))
        return x + self._drop(self.mlp(self.norm2(x)))


class PatchEmbed(nn.Module):
    def __init__(self, img_size, patch, dim):
        super().__init__()
        self.proj = nn.Conv2d(3, dim, patch, patch)
        self.pos = nn.Parameter(torch.zeros(1, (img_size // patch) ** 2, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2) + self.pos


class TokenHead(nn.Module):
    """LayerNorm, mean over tokens, linear classifier."""

    def __init__(self, dim, num_classes):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc = nn.Linear(dim, num_classes)

    def forward(self, x):
        return self.fc(self.norm(x).mean(1))


def _init(m):
    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=0.02)
        nn.init.zeros_(m.bias)


def ee_vit_tiny(num_classes=100, img_size=32, patch=4, dim=192, depth=12, heads=3, exits=(3, 6, 9, 12), drop_path=0.1):
    """ViT-Tiny (patch 4 for 32x32 inputs) with exits after blocks 3, 6, 9 and 12."""
    blocks = [Block(dim, heads, drop_path=drop_path * i / (depth - 1)) for i in range(depth)]
    segments, start = [], 0
    for e in exits:
        segments.append(nn.Sequential(*blocks[start:e]))
        start = e
    segments[0] = nn.Sequential(PatchEmbed(img_size, patch, dim), *segments[0])
    model = MultiExitNet(segments, [TokenHead(dim, num_classes) for _ in exits])
    model.apply(_init)
    return model


MODELS = {"resnet18": resnet18, "ee_resnet18": ee_resnet18, "ee_resnet18_pool": ee_resnet18_pool,
          "ee_vit_tiny": ee_vit_tiny}


def build_model(name, num_classes=100):
    return MODELS[name](num_classes)


import torch
from torch.utils.flop_counter import FlopCounterMode



def _count(module, x):
    with FlopCounterMode(display=False) as fc:
        out = module(x)
    return fc.get_total_flops(), out


@torch.no_grad()
def exit_costs(model, img_size=32):
    """FLOPs per segment and per head, and the cumulative cost of stopping at each exit.

    Stopping at exit k means running segments 0..k and evaluating heads 0..k.
    """
    model = model.eval().cpu()
    h = torch.zeros(1, 3, img_size, img_size)
    seg, head = [], []
    Attention.manual = True
    try:
        for s, hd in zip(model.segments, model.heads):
            f, h = _count(s, h)
            seg.append(f)
            head.append(_count(hd, h)[0])
    finally:
        Attention.manual = False
    cum, total = [], 0
    for s, hd in zip(seg, head):
        total += s + hd
        cum.append(total)
    return {"segment": seg, "head": head, "cumulative": cum}


def count_params(model):
    return sum(p.numel() for p in model.parameters())


In [ ]:
for name in MODELS:
    m = build_model(name)
    c = exit_costs(m)
    print(f"{name:17s} {count_params(m) / 1e6:5.2f} M params | GFLOPs to reach each exit:",
          [round(f / 1e9, 3) for f in c["cumulative"]])

## 3. Exit rules, calibration and early-exit inference

* **Max-prob** (Model 2): exit when the largest softmax probability is at least a threshold.
* **Entropy**: exit when 1 - H(p)/log(100) is at least a threshold.
* **Temperature scaling** (Model 3): divide each exit's logits by a temperature fitted on validation data.
* Thresholds are chosen on validation: the cheapest one that loses no accuracy relative to the model's own final
  exit (one global threshold, or one per exit by coordinate descent).
* `adaptive_forward` is real early-exit inference: images that exit are removed from the batch, so the saving is
  real computation.

In [ ]:
"""Exit criteria, temperature scaling, calibration metrics and early-exit inference."""
import math

import numpy as np
import torch
import torch.nn.functional as F


def max_prob(logits, t=1.0):
    return F.softmax(logits.float() / t, -1).amax(-1)


def neg_entropy(logits, t=1.0):
    """1 - H(p) / log(C), so that larger means more confident, as with max_prob."""
    logp = F.log_softmax(logits.float() / t, -1)
    h = -(logp.exp() * logp).sum(-1)
    return 1 - h / math.log(logits.shape[-1])


SCORES = {"max_prob": max_prob, "entropy": neg_entropy}


def fit_temperature(logits, labels, iters=200):
    """Single temperature minimizing validation NLL (Guo et al., 2017)."""
    logits = logits.float()
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.05, max_iter=iters, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / log_t.exp(), labels)
        loss.backward()
        return loss

    opt.step(closure)
    return log_t.exp().item()


def reliability(logits, labels, t=1.0, bins=15):
    """Per-bin (confidence, accuracy, fraction) and the expected calibration error."""
    conf, pred = F.softmax(logits.float() / t, -1).max(-1)
    correct = pred.eq(labels).float()
    edges = torch.linspace(0, 1, bins + 1)
    rows, ece = [], 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            frac = m.float().mean().item()
            c, a = conf[m].mean().item(), correct[m].mean().item()
            ece += frac * abs(c - a)
            rows.append((c, a, frac))
    return rows, ece


def ece(logits, labels, t=1.0, bins=15):
    return reliability(logits, labels, t, bins)[1]


def assign_exits(logits, score, thresh, temps=None):
    """Offline routing. logits: [K, N, C]. Returns the exit index taken by each sample."""
    k_total, n, _ = logits.shape
    temps = temps or [1.0] * k_total
    exit_at = torch.full((n,), k_total - 1, dtype=torch.long)
    pending = torch.ones(n, dtype=torch.bool)
    for k in range(k_total - 1):
        hit = pending & (score(logits[k], temps[k]) >= thresh)
        exit_at[hit] = k
        pending &= ~hit
    return exit_at


def simulate(logits, labels, costs, score, thresh, temps=None):
    """Accuracy, mean cost and exit histogram of a threshold policy, computed from stored logits."""
    exit_at = assign_exits(logits, score, thresh, temps)
    n = labels.shape[0]
    pred = logits.argmax(-1)[exit_at, torch.arange(n)]
    costs = torch.as_tensor(costs, dtype=torch.float64)
    return {
        "thresh": float(thresh),
        "acc": pred.eq(labels).float().mean().item(),
        "cost": costs[exit_at].mean().item(),
        "exit_frac": torch.bincount(exit_at, minlength=logits.shape[0]).div(n).tolist(),
    }


@torch.no_grad()
def adaptive_forward(model, x, score, thresh, temps=None, force_exit=None):
    """Real early-exit inference: samples that exit are dropped from the batch before the next segment."""
    k_total = model.num_exits
    temps = temps or [1.0] * k_total
    last = k_total - 1 if force_exit is None else force_exit
    n = x.shape[0]
    preds = torch.empty(n, dtype=torch.long, device=x.device)
    exit_at = torch.full((n,), last, dtype=torch.long, device=x.device)
    alive = torch.arange(n, device=x.device)
    h = x
    for k in range(last + 1):
        h = model.segments[k](h)
        logits = model.heads[k](h)
        if k == last:
            preds[alive] = logits.argmax(1)
            break
        done = score(logits, temps[k]) >= thresh
        if force_exit is not None:
            done = torch.zeros_like(done)
        if done.any():
            preds[alive[done]] = logits[done].argmax(1)
            exit_at[alive[done]] = k
            keep = ~done
            if not keep.any():
                break
            alive, h = alive[keep], h[keep]
    return preds, exit_at


THRESH = np.unique(np.concatenate([np.linspace(0, 0.9, 91), np.linspace(0.9, 1.0, 201), [1.01]]))
POLICIES = [("max_prob", False, "Max-prob"), ("entropy", False, "Entropy"),
            ("max_prob", True, "Max-prob + TS"), ("entropy", True, "Entropy + TS")]


def exit_scores(logits, policy, calibrated, temps):
    """Scores of the non-final exits as a [K-1, N] array."""
    k = logits.shape[0]
    return np.stack([SCORES[policy](logits[i], temps[i] if calibrated else 1.0).numpy() for i in range(k - 1)])


def route(scores, th):
    """First exit whose score clears its threshold (one value, or one per exit), else the last exit."""
    th = np.asarray(th, dtype=float)
    if th.ndim:
        th = th[:, None]
    hit = np.vstack([scores >= th, np.ones((1, scores.shape[1]), bool)])
    return hit.argmax(0)


def routing_stats(exit_at, correct, costs):
    k, n = correct.shape
    return {"acc": float(correct[exit_at, np.arange(n)].mean()), "cost": float(np.asarray(costs)[exit_at].mean()),
            "exit_frac": (np.bincount(exit_at, minlength=k) / n).tolist()}


def sweep(scores, correct, costs, grid=THRESH):
    return [dict(thresh=float(t), **routing_stats(route(scores, t), correct, costs)) for t in grid]


def pick_threshold(rows, target):
    """Cheapest threshold whose accuracy reaches the target."""
    ok = [r for r in rows if r["acc"] >= target]
    return min(ok, key=lambda r: (r["cost"], -r["acc"]))["thresh"]


def tune_per_exit(scores, correct, costs, target, init, grid=THRESH, passes=3):
    """Coordinate descent over one threshold per exit, minimising cost subject to accuracy >= target."""
    th = np.full(scores.shape[0], float(init))
    best = routing_stats(route(scores, th), correct, costs)
    for _ in range(passes):
        for k in range(len(th)):
            for t in grid:
                cand = th.copy()
                cand[k] = t
                r = routing_stats(route(scores, cand), correct, costs)
                if r["acc"] >= target and (r["cost"], -r["acc"]) < (best["cost"], -best["acc"]):
                    th, best = cand, r
    return th.tolist(), best


## 4. Training

All exits are trained jointly with an equally weighted loss. With self-distillation weight alpha, each early exit
minimises (1 - alpha) CE + alpha tau^2 KL(final exit || exit) at temperature tau = 3, with the final exit
detached. ResNets use SGD; the ViT uses AdamW with label smoothing, mixup/CutMix and RandAugment. Mixed precision
(fp16) is used on the GPU.

In [ ]:
import argparse
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn.functional as F



def train_main_args():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--data", default="data")
    p.add_argument("--epochs", type=int, default=80)
    p.add_argument("--bs", type=int, default=128)
    p.add_argument("--opt", default="sgd", choices=["sgd", "adamw"])
    p.add_argument("--lr", type=float, default=0.1)
    p.add_argument("--wd", type=float, default=5e-4)
    p.add_argument("--warmup", type=int, default=2, help="warmup epochs")
    p.add_argument("--label-smoothing", type=float, default=0.0)
    p.add_argument("--mix", action="store_true", help="mixup or cutmix on every batch")
    p.add_argument("--clip", type=float, default=0.0)
    p.add_argument("--distill", type=float, default=0.0, help="weight of self-distillation from the final exit")
    p.add_argument("--kd-temp", type=float, default=3.0)
    p.add_argument("--randaug", action="store_true")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--device", default="cuda")
    p.add_argument("--limit", type=int, default=0, help="train on the first N images (smoke tests)")
    return p.parse_args()


def mix_batch(x, y):
    perm = torch.randperm(x.shape[0], device=x.device)
    if np.random.rand() < 0.5:
        lam = float(np.random.beta(0.8, 0.8))
        return lam * x + (1 - lam) * x[perm], y, y[perm], lam
    lam = float(np.random.beta(1.0, 1.0))
    h, w = x.shape[2:]
    rh, rw = int(h * math.sqrt(1 - lam)), int(w * math.sqrt(1 - lam))
    cy, cx = np.random.randint(h), np.random.randint(w)
    y0, y1, x0, x1 = max(cy - rh // 2, 0), min(cy + rh // 2, h), max(cx - rw // 2, 0), min(cx + rw // 2, w)
    x = x.clone()
    x[:, :, y0:y1, x0:x1] = x[perm, :, y0:y1, x0:x1]
    return x, y, y[perm], 1 - (y1 - y0) * (x1 - x0) / (h * w)


def multi_exit_loss(outs, ya, yb, lam, ls, alpha=0.0, tau=3.0):
    """Equal-weight average over exits; early exits optionally also match the final exit (self-distillation)."""
    def ce(o):
        loss = lam * F.cross_entropy(o, ya, label_smoothing=ls)
        if lam < 1:
            loss = loss + (1 - lam) * F.cross_entropy(o, yb, label_smoothing=ls)
        return loss

    teacher = F.softmax(outs[-1].detach().float() / tau, -1)
    total = ce(outs[-1])
    for o in outs[:-1]:
        loss = ce(o)
        if alpha:
            kd = F.kl_div(F.log_softmax(o.float() / tau, -1), teacher, reduction="batchmean") * tau ** 2
            loss = (1 - alpha) * loss + alpha * kd
        total = total + loss
    return total / len(outs)


@torch.no_grad()
def evaluate(model, loader, amp):
    model.eval()
    correct, n = None, 0
    for x, y in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            outs = model(x)
        c = torch.stack([o.argmax(1).eq(y).sum() for o in outs])
        correct = c if correct is None else correct + c
        n += len(y)
    return (correct.float() / n).tolist()


def train_main():
    args = train_main_args()
    os.makedirs(args.out, exist_ok=True)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    dev = torch.device(args.device)
    amp = dev.type == "cuda"
    torch.backends.cudnn.benchmark = True

    data = load_cifar100(args.data)
    xtr, ytr = to_device(data["train"], dev)
    if args.limit:
        xtr, ytr = xtr[:args.limit], ytr[:args.limit]
    xva, yva = to_device(data["val"], dev)
    train_loader = GPULoader(xtr, ytr, args.bs, train=True, transform=ChunkRandAugment() if args.randaug else None)
    val_loader = GPULoader(xva, yva, 500)

    model = build_model(args.model).to(dev)
    decay = [p for n, p in model.named_parameters() if p.ndim > 1 and "pos" not in n]
    no_decay = [p for n, p in model.named_parameters() if p.ndim <= 1 or "pos" in n]
    groups = [{"params": decay, "weight_decay": args.wd}, {"params": no_decay, "weight_decay": 0.0}]
    if args.opt == "sgd":
        opt = torch.optim.SGD(groups, lr=args.lr, momentum=0.9, nesterov=True)
    else:
        opt = torch.optim.AdamW(groups, lr=args.lr, betas=(0.9, 0.999))
    steps = args.epochs * len(train_loader)
    warm = args.warmup * len(train_loader)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    history = []
    for epoch in range(args.epochs):
        model.train()
        t0, tot, nb = time.time(), 0.0, 0
        for x, y in train_loader:
            ya, yb, lam = y, y, 1.0
            if args.mix:
                x, ya, yb, lam = mix_batch(x, y)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                loss = multi_exit_loss(model(x), ya, yb, lam, args.label_smoothing, args.distill, args.kd_temp)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if args.clip:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item()
            nb += 1
        val_acc = evaluate(model, val_loader, amp)
        rec = {"epoch": epoch + 1, "loss": tot / nb, "val_acc": val_acc, "lr": sched.get_last_lr()[0],
               "time": time.time() - t0}
        history.append(rec)
        print(json.dumps(rec), flush=True)
        if (epoch + 1) % 10 == 0 or epoch + 1 == args.epochs:
            torch.save({"model": args.model, "args": vars(args), "state_dict": model.state_dict()},
                       os.path.join(args.out, "model.pt"))
            with open(os.path.join(args.out, "history.json"), "w") as f:
                json.dump(history, f)


## 5. Evaluation code

Logits of every exit are stored for the validation set, the clean test set and all 30 corrupted test sets, so every exit rule can be analysed offline. Latency is measured with real early-exit inference, one image at a time, on the GPU and on the CPU.

In [ ]:
"""Latency of trained runs measured on one idle machine: batch-1 GPU and CPU, batched GPU throughput."""
import argparse
import copy
import glob
import json
import os
import platform
import re
import time

import numpy as np
import torch


GRID = (0.6, 0.8, 0.9, 0.95)
TOLS = {"strict": 0.0, "relaxed": 0.01}


def _sync(dev):
    if dev.type == "cuda":
        torch.cuda.synchronize(dev)


def batch1_latency(model, xs, dev, **kw):
    """Mean milliseconds per image when images arrive one at a time."""
    for i in range(min(20, len(xs))):
        adaptive_forward(model, xs[i:i + 1], **kw)
    _sync(dev)
    preds, exits = [], []
    t0 = time.perf_counter()
    for i in range(len(xs)):
        p, e = adaptive_forward(model, xs[i:i + 1], **kw)
        preds.append(p)
        exits.append(e)
    _sync(dev)
    ms = (time.perf_counter() - t0) * 1000 / len(xs)
    return ms, torch.cat(preds).cpu(), torch.cat(exits).cpu()


def throughput(model, xs, dev, bs=256, **kw):
    for i in range(0, min(len(xs), 4 * bs), bs):
        adaptive_forward(model, xs[i:i + bs], **kw)
    _sync(dev)
    t0 = time.perf_counter()
    for i in range(0, len(xs), bs):
        adaptive_forward(model, xs[i:i + bs], **kw)
    _sync(dev)
    return len(xs) / (time.perf_counter() - t0)


def cpu_name():
    try:
        with open("/proc/cpuinfo") as f:
            return next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
    except (OSError, StopIteration):
        return platform.processor()


def operating_points(run, metrics, tol):
    """Thresholds selected on validation for each policy, with the same rule as the analysis."""
    lv = torch.from_numpy(np.load(os.path.join(run, "logits", "val.npy")).astype(np.float32))
    correct = lv.argmax(-1).numpy() == np.load(os.path.join(run, "logits", "val_labels.npy"))
    costs = metrics["flops"]["cumulative"]
    target = float(correct[-1].mean()) - tol
    return {label: pick_threshold(sweep(exit_scores(lv, p, cal, metrics["temperature"]), correct, costs), target)
            for p, cal, label in POLICIES}


def configs(ops):
    rows = [{"label": label, "policy": p, "calibrated": cal, "thresh": ops[point][label], "point": point}
            for point in TOLS for p, cal, label in POLICIES]
    rows += [{"label": "Max-prob", "policy": "max_prob", "calibrated": False, "thresh": t, "point": "grid"} for t in GRID]
    return rows


def _kw(row, temps):
    return {"score": SCORES[row["policy"]], "thresh": row["thresh"], "temps": temps if row["calibrated"] else None}


def latency_suite(model, xs, ys, dev, rows, temps):
    k = model.num_exits
    res = {"forced_ms": [batch1_latency(model, xs, dev, score=SCORES["max_prob"], thresh=2.0, force_exit=i)[0]
                         for i in range(k)], "adaptive": []}
    for row in rows:
        ms, preds, exits = batch1_latency(model, xs, dev, **_kw(row, temps))
        res["adaptive"].append({**row, "ms": ms, "acc": preds.eq(ys.cpu()).float().mean().item(),
                                "exit_frac": torch.bincount(exits, minlength=k).div(len(exits)).tolist()})
    return res


def throughput_suite(model, xs, dev, rows, temps):
    res = {"full": throughput(model, xs, dev, score=SCORES["max_prob"], thresh=2.0), "adaptive": []}
    for row in rows:
        res["adaptive"].append({**row, "img_per_s": throughput(model, xs, dev, **_kw(row, temps))})
    return res


def latency_main():
    p = argparse.ArgumentParser()
    p.add_argument("--runs", nargs="+", required=True, help="roots searched recursively for metrics.json")
    p.add_argument("--only", default=".*", help="regex on the run directory name")
    p.add_argument("--out", required=True)
    p.add_argument("--data", default="data")
    p.add_argument("--device", default="cuda")
    p.add_argument("--gpu-images", type=int, default=2000)
    p.add_argument("--cpu-images", type=int, default=500)
    p.add_argument("--threads", type=int, nargs="+", default=[1, 4])
    args = p.parse_args()
    dev = torch.device(args.device)
    found = {os.path.dirname(m) for g in args.runs for m in glob.glob(os.path.join(g, "**", "metrics.json"), recursive=True)}
    runs = sorted((r for r in found if re.search(args.only, os.path.basename(r))), key=os.path.basename)
    print("runs:", runs, flush=True)
    if not runs:
        for g in args.runs:
            for root, dirs, _ in os.walk(g):
                if root.count(os.sep) - g.count(os.sep) < 4:
                    print("dir:", root, flush=True)
    data = load_cifar100(args.data)
    x, y = to_device(data["test"], dev)
    xs = next(iter(GPULoader(x, y, len(y))))[0]
    for run in runs:
        with open(os.path.join(run, "metrics.json")) as f:
            metrics = json.load(f)
        ck = torch.load(os.path.join(run, "model.pt"), map_location="cpu")
        model = build_model(ck["model"])
        model.load_state_dict(ck["state_dict"])
        model.eval().to(dev)
        temps = metrics["temperature"]
        ops = {point: operating_points(run, metrics, tol) for point, tol in TOLS.items()} if model.num_exits > 1 else {}
        rows = configs(ops) if model.num_exits > 1 else []
        out = {"run": os.path.basename(run), "model": ck["model"], "ops": ops}
        if dev.type == "cuda":
            n = args.gpu_images
            out["gpu"] = latency_suite(model, xs[:n], y[:n], dev, rows, temps)
            out["gpu"]["batched"] = throughput_suite(model, xs, dev, rows, temps)
            out["gpu"]["device"] = torch.cuda.get_device_name(dev)
        cpu_model = copy.deepcopy(model).cpu()
        n = args.cpu_images
        for t in args.threads:
            torch.set_num_threads(t)
            out[f"cpu{t}"] = latency_suite(cpu_model, xs[:n].cpu(), y[:n].cpu(), torch.device("cpu"), rows, temps)
            out[f"cpu{t}"]["device"] = f"{cpu_name()} ({t} thread{'s' if t > 1 else ''})"
        d = os.path.join(args.out, os.path.basename(run))
        os.makedirs(d, exist_ok=True)
        with open(os.path.join(d, "latency.json"), "w") as f:
            json.dump(out, f, indent=1)
        print(out["run"], {k: v["forced_ms"] for k, v in out.items() if isinstance(v, dict) and "forced_ms" in v},
              flush=True)


In [ ]:
import argparse
import copy
import json
import os

import numpy as np
import torch


SEVERITIES = (1, 2, 3, 4, 5)
LAT_THRESH = (0.6, 0.8, 0.9, 0.95)
LAT_POLICIES = (("max_prob", False), ("max_prob", True), ("entropy", True))


def evaluate_main_args():
    p = argparse.ArgumentParser()
    p.add_argument("--run", required=True, help="training output dir containing model.pt")
    p.add_argument("--data", default="data")
    p.add_argument("--device", default="cuda")
    p.add_argument("--gpu-images", type=int, default=2000)
    p.add_argument("--cpu-images", type=int, default=300)
    p.add_argument("--cpu-threads", type=int, default=1)
    p.add_argument("--no-corrupt", action="store_true")
    p.add_argument("--corrupt-val", action="store_true", help="also store corrupted validation logits")
    p.add_argument("--corrupt-val-only", action="store_true", help="only compute corrupted validation logits")
    p.add_argument("--out", help="output dir (default: --run)")
    p.add_argument("--no-latency", action="store_true")
    return p.parse_args()


@torch.no_grad()
def predict(model, x, y, transform=None):
    """Logits of every exit as a [K, N, C] float16 CPU tensor."""
    return torch.cat([torch.stack(model(xb)).half().cpu() for xb, _ in GPULoader(x, y, 500, transform=transform)], 1)


def corrupt_sets(model, x, y, logit_dir, dev, prefix="", seed=0):
    """Logits on every corruption and severity, saved as <prefix><name>_<s>.npy; returns per-exit accuracy."""
    acc = {}
    for ci, name in enumerate(CORRUPTIONS):
        acc[name] = {}
        for s in SEVERITIES:
            g = torch.Generator(device=dev).manual_seed(seed + 1000 * ci + s)
            logits = predict(model, x, y, transform=corrupt(name, s, g))
            np.save(os.path.join(logit_dir, f"{prefix}{name}_{s}.npy"), logits.numpy())
            acc[name][s] = per_exit_acc(logits, y.cpu())
        print(prefix + name, json.dumps(acc[name]), flush=True)
    return acc


def per_exit_acc(logits, labels):
    return logits.argmax(-1).eq(labels).float().mean(1).tolist()


def _policies(temps):
    for name, calibrated in LAT_POLICIES:
        for th in LAT_THRESH:
            yield {"policy": name, "calibrated": calibrated, "thresh": th}, \
                {"score": SCORES[name], "thresh": th, "temps": temps if calibrated else None}


def eval_latency_suite(model, xs, ys, dev, temps):
    """Batch-1 latency of every forced exit and of each threshold policy."""
    k_total = model.num_exits
    res = {"forced_ms": [], "adaptive": []}
    for k in range(k_total):
        res["forced_ms"].append(batch1_latency(model, xs, dev, score=SCORES["max_prob"], thresh=2.0, force_exit=k)[0])
    if k_total > 1:
        for row, kw in _policies(temps):
            ms, preds, exits = batch1_latency(model, xs, dev, **kw)
            row.update(ms=ms, acc=preds.eq(ys.cpu()).float().mean().item(),
                       exit_frac=torch.bincount(exits, minlength=k_total).div(len(exits)).tolist())
            res["adaptive"].append(row)
    return res


def eval_throughput_suite(model, xs, dev, temps):
    """Images per second with batches of 256, exited samples removed from the batch."""
    res = {"full": throughput(model, xs, dev, score=SCORES["max_prob"], thresh=2.0), "adaptive": []}
    if model.num_exits > 1:
        for row, kw in _policies(temps):
            row["img_per_s"] = throughput(model, xs, dev, **kw)
            res["adaptive"].append(row)
    return res


def evaluate_main():
    args = evaluate_main_args()
    dev = torch.device(args.device)
    ck = torch.load(os.path.join(args.run, "model.pt"), map_location="cpu")
    model = build_model(ck["model"])
    model.load_state_dict(ck["state_dict"])
    model.eval()
    out = {"model": ck["model"], "train_args": ck["args"], "params": count_params(model),
           "flops": exit_costs(model), "commit": os.environ.get("GIT_COMMIT", "")}
    model.to(dev)

    data = load_cifar100(args.data)
    out_dir = args.out or args.run
    logit_dir = os.path.join(out_dir, "logits")
    os.makedirs(logit_dir, exist_ok=True)
    if args.corrupt_val_only:
        xv, yv = to_device(data["val"], dev)
        acc = corrupt_sets(model, xv, yv, logit_dir, dev, prefix="val_", seed=50000)
        with open(os.path.join(out_dir, "corrupt_val.json"), "w") as f:
            json.dump(acc, f)
        return
    splits = {}
    for split in ("val", "test"):
        x, y = to_device(data[split], dev)
        logits = predict(model, x, y)
        splits[split] = (x, y, logits)
        np.save(os.path.join(logit_dir, f"{split}.npy"), logits.numpy())
        np.save(os.path.join(logit_dir, f"{split}_labels.npy"), y.cpu().numpy())
    yv, yt = splits["val"][1].cpu(), splits["test"][1].cpu()
    lv, lt = splits["val"][2], splits["test"][2]
    out["acc"] = {"val": per_exit_acc(lv, yv), "test": per_exit_acc(lt, yt)}
    temps = [fit_temperature(lv[k], yv) for k in range(model.num_exits)]
    out["temperature"] = temps
    out["ece_test"] = {"raw": [ece(lt[k], yt) for k in range(len(temps))],
                       "calibrated": [ece(lt[k], yt, temps[k]) for k in range(len(temps))]}
    print(json.dumps({k: out[k] for k in ("acc", "temperature", "ece_test")}), flush=True)

    if not args.no_corrupt:
        out["corrupt_acc"] = corrupt_sets(model, splits["test"][0], splits["test"][1], logit_dir, dev)
    if args.corrupt_val:
        out["corrupt_val_acc"] = corrupt_sets(model, splits["val"][0], splits["val"][1], logit_dir, dev,
                                              prefix="val_", seed=50000)

    if not args.no_latency:
        x, y = splits["test"][0], splits["test"][1]
        xs = next(iter(GPULoader(x, y, len(y))))[0]
        lat = {}
        if dev.type == "cuda":
            lat["gpu"] = eval_latency_suite(model, xs[:args.gpu_images], y[:args.gpu_images], dev, temps)
            lat["gpu"]["batched"] = eval_throughput_suite(model, xs, dev, temps)
            lat["gpu"]["device"] = torch.cuda.get_device_name(dev)
        torch.set_num_threads(args.cpu_threads)
        cpu_model = copy.deepcopy(model).cpu()
        n = args.cpu_images
        lat["cpu"] = eval_latency_suite(cpu_model, xs[:n].cpu(), y[:n].cpu(), torch.device("cpu"), temps)
        lat["cpu"]["device"] = f"{cpu_name()} ({args.cpu_threads} thread)"
        out["latency"] = lat

    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(out, f, indent=1)
    print("saved", os.path.join(out_dir, "metrics.json"), flush=True)


## 6. Train and evaluate

The three models of the report, with the settings of the final runs. A run whose `metrics.json` exists is
skipped, so with Google Drive mounted you can re-run this cell after a disconnect. Add
`("ee_resnet18_ce", "ee_resnet18", RESNET)` for the no-distillation ablation or
`("ee_resnet18_pool", "ee_resnet18_pool", RESNET + KD)` for the pooling-head ablation.

In [ ]:
EPOCHS = {"full": 200, "fast": 30, "smoke": 1}[MODE]
RESNET = ["--epochs", EPOCHS, "--bs", 128, "--opt", "sgd", "--lr", 0.1, "--wd", 5e-4, "--warmup", 2 if EPOCHS > 2 else 0]
VIT = ["--epochs", EPOCHS, "--bs", 256, "--opt", "adamw", "--lr", 1e-3, "--wd", 0.05, "--warmup", 5 if EPOCHS > 5 else 0,
       "--label-smoothing", 0.1, "--mix", "--clip", 1.0, "--randaug"]
KD = ["--distill", 0.5, "--kd-temp", 3]
JOBS = [("resnet18", "resnet18", RESNET),              # Model 1
        ("ee_resnet18_kd", "ee_resnet18", RESNET + KD),  # Models 2 and 3
        ("ee_vit_tiny", "ee_vit_tiny", VIT + KD)]        # Model 4
SMOKE = ["--bs", 64, "--limit", 256] if MODE == "smoke" else []

for seed in SEEDS:
    for name, model, args in JOBS:
        out = os.path.join(RUNS_DIR, "final-colab", f"{name}_s{seed}")
        if os.path.exists(os.path.join(out, "metrics.json")):
            print("done:", out); continue
        t0 = time.time()
        cli(train_main, "--model", model, "--seed", seed, "--out", out, "--data", DATA, "--device", DEVICE, *args, *SMOKE)
        cli(evaluate_main, "--run", out, "--data", DATA, "--device", DEVICE, "--no-latency")
        print(f"{name} seed {seed}: {(time.time() - t0) / 60:.1f} min")

## 7. Latency

Time per image when images arrive one at a time (batch 1) on the GPU and on one and four CPU threads, for every
seed-0 model at its validation-selected thresholds, plus GPU throughput with batches of 256.

In [ ]:
cli(latency_main, "--runs", os.path.join(RUNS_DIR, "final-colab"), "--only", "_s0$", "--out", os.path.join(RUNS_DIR, "latency"),
    "--data", DATA, "--device", DEVICE, *(["--gpu-images", 50, "--cpu-images", 20, "--threads", 1] if MODE == "smoke" else []))

## 8. Analysis

Threshold sweeps for the four exit rules, operating points chosen on validation (global and per-exit thresholds),
calibration, overthinking, latency and robustness under the 30 corruption settings, averaged over seeds.

In [ ]:
import matplotlib

BLUE, ORANGE, AQUA, YELLOW, VIOLET, RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#4a3aa7", "#e34948"
INK, INK2, GRID_COLOR = "#0b0b0b", "#52514e", "#e4e3df"
POLICY_STYLE = {"Max-prob": (BLUE, "-"), "Entropy": (AQUA, "--"), "Max-prob + TS": (ORANGE, "-."),
                "Entropy + TS": (YELLOW, ":")}
EXIT_RAMP = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID_COLOR,
    "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False, "lines.linewidth": 1.6, "lines.markersize": 5,
    "font.family": "DejaVu Sans",
})


def savefig(fig, out, name):
    fig.tight_layout()
    fig.savefig(os.path.join(out, "figures", name), bbox_inches="tight", facecolor="white")
    plt.close(fig)


In [ ]:
"""Final analysis over seeds: operating points, per-exit thresholds, ablations, calibration, corruptions, latency.

Reads runs/final-*/<run>/ and runs/latency/<run>/latency.json, writes results/final/.
"""
import argparse
import glob
import itertools
import json
import os
import re
import shutil
import sys

import numpy as np
import torch



SEV_LABELS = ("1", "2", "3", "4", "5")
TOL, TOL_RELAXED = 0.0, 0.01
NAMES = {"resnet18": "ResNet-18 (full)", "ee_resnet18_kd": "EE-ResNet-18",
         "ee_resnet18_ce": "EE-ResNet-18, no distillation", "ee_resnet18_pool": "EE-ResNet-18, pooling heads",
         "ee_vit_tiny": "EE-ViT-Tiny"}
YELLOW_SAFE = POLICY_STYLE["Entropy + TS"][0]
CNAMES = {"gaussian_noise": "Gaussian noise", "shot_noise": "Shot noise", "impulse_noise": "Impulse noise",
          "gaussian_blur": "Gaussian blur", "defocus_blur": "Defocus blur", "contrast": "Contrast"}


def ms(values):
    v = np.asarray([x for x in values if x is not None], dtype=float)
    return [float(np.nanmean(v)), float(np.nanstd(v))] if len(v) else [float("nan"), float("nan")]


def load(d, name):
    return torch.from_numpy(np.load(os.path.join(d, "logits", f"{name}.npy")).astype(np.float32))


def labels(d, split):
    return np.load(os.path.join(d, "logits", f"{split}_labels.npy"))


def discover(root):
    runs = {}
    for m in sorted(glob.glob(os.path.join(root, "final-*", "*", "metrics.json"))):
        d = os.path.dirname(m)
        name = os.path.basename(d)
        cfg, seed = re.match(r"(.+)_s(\d+)$", name).groups()
        with open(m) as f:
            metrics = json.load(f)
        with open(os.path.join(d, "history.json")) as f:
            metrics["history"] = json.load(f)
        metrics.update(dir=d, name=name, cfg=cfg, seed=int(seed))
        lat = os.path.join(root, "latency", name, "latency.json")
        if os.path.exists(lat):
            with open(lat) as f:
                metrics["latency"] = json.load(f)
        runs.setdefault(cfg, []).append(metrics)
    return runs


def subset_costs(seg, head, exits):
    """Cumulative cost of stopping at each exit in `exits` when only those heads are evaluated."""
    return [sum(seg[:e + 1]) + sum(head[j] for j in exits if j <= e) for e in exits]


def analyze_run(r, ref_cost, match_acc):
    d, k = r["dir"], len(r["acc"]["test"])
    out = {"seed": r["seed"], "test_acc": r["acc"]["test"], "val_acc": r["acc"]["val"],
           "gflops": [c / 1e9 for c in r["flops"]["cumulative"]], "params": r["params"],
           "temperature": r["temperature"], "ece_test": r["ece_test"]}
    if k == 1:
        out["corrupt_acc"] = {c: {s: r["corrupt_acc"][c][s][0] for s in SEV_LABELS} for c in CORRUPTIONS}
        return out
    costs, temps = out["gflops"], r["temperature"]
    lv, lt = load(d, "val"), load(d, "test")
    yv, yt = labels(d, "val"), labels(d, "test")
    cv, ct = lv.argmax(-1).numpy() == yv, lt.argmax(-1).numpy() == yt
    final_val = float(cv[-1].mean())  # from the stored logits, so the final exit always meets the target
    target, relaxed = final_val - TOL, final_val - TOL_RELAXED
    match_acc = match_acc if match_acc is not None else out["test_acc"][-1]
    budgets = [round(f * ref_cost, 4) for f in (0.35, 0.5, 0.65, 0.8)]
    out["budgets"] = budgets
    out["policies"], out["curves"] = {}, {}
    for policy, cal, label in POLICIES:
        sv, st = exit_scores(lv, policy, cal, temps), exit_scores(lt, policy, cal, temps)
        sw_v, sw_t = sweep(sv, cv, costs), sweep(st, ct, costs)
        th = pick_threshold(sw_v, target)
        res = {"thresh": th, "test": routing_stats(route(st, th), ct, costs),
               "acc_at": [max([x["acc"] for x in sw_t if x["cost"] <= b + 1e-9], default=float("nan")) for b in budgets]}
        th_k, _ = tune_per_exit(sv, cv, costs, target, th)
        res["per_exit"] = {"thresh": th_k, "test": routing_stats(route(st, th_k), ct, costs)}
        th_r = pick_threshold(sw_v, relaxed)
        res["relaxed"] = {"thresh": th_r, "test": routing_stats(route(st, th_r), ct, costs)}
        res["match_cost"] = min([x["cost"] for x in sw_t if x["acc"] >= match_acc], default=None)
        out["policies"][label] = res
        out["curves"][label] = [(x["cost"], x["acc"]) for x in sw_t]
    first = np.where(ct.any(0), ct.argmax(0), k - 1)
    out["oracle"] = {"acc": float(ct.any(0).mean()), "cost": float(np.asarray(costs)[first].mean())}
    out["overthinking"] = {"destructive": float((ct[:-1].any(0) & ~ct[-1]).mean()),
                           "early_correct": [float(ct[i].mean()) for i in range(k)]}
    seg, head = [f / 1e9 for f in r["flops"]["segment"]], [f / 1e9 for f in r["flops"]["head"]]
    out["subsets"] = {}
    for n in range(k):
        for s in itertools.combinations(range(k - 1), n):
            exits = [*s, k - 1]
            c = subset_costs(seg, head, exits)
            if n:
                sv = exit_scores(lv[exits], "max_prob", False, [1.0] * len(exits))
                st = exit_scores(lt[exits], "max_prob", False, [1.0] * len(exits))
                th = pick_threshold(sweep(sv, cv[exits], c), target)
            else:
                st, th = np.zeros((0, lt.shape[1])), 2.0
            out["subsets"]["+".join(f"E{e + 1}" for e in exits)] = routing_stats(route(st, th), ct[exits], c)
    out["corrupt"], out["corrupt_ece"] = {}, {}
    for c in CORRUPTIONS:
        out["corrupt"][c], out["corrupt_ece"][c] = {}, {}
        for s in SEV_LABELS:
            lg = load(d, f"{c}_{s}")
            cc = lg.argmax(-1).numpy() == yt
            row = {"final_acc": float(cc[-1].mean())}
            for policy, cal, label in POLICIES:
                ex = route(exit_scores(lg, policy, cal, temps), out["policies"][label]["thresh"])
                m = routing_stats(ex, cc, costs)
                early = ex < k - 1
                m["early_frac"] = float(early.mean())
                m["early_acc"] = float(cc[ex, np.arange(len(ex))][early].mean()) if early.any() else None
                row[label] = m
            out["corrupt"][c][s] = row
            yt_t = torch.from_numpy(yt)
            out["corrupt_ece"][c][s] = {"raw": [reliability(lg[i], yt_t)[1] for i in range(k)],
                                        "calibrated": [reliability(lg[i], yt_t, temps[i])[1] for i in range(k)]}
    return out


def aggregate(per_seed):
    """Mean and std over seeds of every numeric leaf that all seeds share."""
    first = per_seed[0]
    if isinstance(first, dict):
        return {key: aggregate([p[key] for p in per_seed]) for key in first if all(key in p for p in per_seed)}
    if isinstance(first, list) and first and not isinstance(first[0], (list, tuple, dict)):
        return [ms([p[i] for p in per_seed]) for i in range(len(first))]
    if isinstance(first, (int, float)) or first is None:
        return ms(per_seed)
    return first


def mean_curve(curves, grid):
    ys = []
    for cur in curves:
        ys.append([max([a for c, a in cur if c <= b + 1e-9], default=np.nan) for b in grid])
    ys = np.asarray(ys)
    return np.nanmean(ys, 0), np.nanstd(ys, 0)


def fig_tradeoff(A, out):
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.9))
    base = A["resnet18"]["mean"]
    for ax, cfg in zip(axes, ("ee_resnet18_kd", "ee_vit_tiny")):
        if cfg not in A:
            ax.set_visible(False)
            continue
        seeds, m = A[cfg]["seeds"], A[cfg]["mean"]
        lo = min(x[0] for x in seeds[0]["curves"]["Max-prob"])
        hi = max(x[0] for x in seeds[0]["curves"]["Max-prob"])
        grid = np.linspace(lo, hi, 160)
        for label, (color, ls) in POLICY_STYLE.items():
            mu, sd = mean_curve([s["curves"][label] for s in seeds], grid)
            ax.plot(grid, 100 * mu, color=color, ls=ls, label=label)
            ax.fill_between(grid, 100 * (mu - sd), 100 * (mu + sd), color=color, alpha=0.12, lw=0)
            op = m["policies"][label]["test"]
            ax.plot(op["cost"][0], 100 * op["acc"][0], "o", color=color, ms=6, mec="white", mew=1.2, zorder=5)
        g, acc = m["gflops"], m["test_acc"]
        ax.errorbar([x[0] for x in g], [100 * a[0] for a in acc], yerr=[100 * a[1] for a in acc], fmt="s",
                    color=INK2, ms=4.5, capsize=2, label="Single exit (forced)")
        for i, (c, a) in enumerate(zip(g, acc)):
            ax.annotate(f"E{i + 1}", (c[0], 100 * a[0]), textcoords="offset points", xytext=(5, -10), fontsize=7,
                        color=INK2)
        if cfg == "ee_resnet18_kd":
            ax.errorbar(base["gflops"][0][0], 100 * base["test_acc"][0][0], yerr=100 * base["test_acc"][0][1],
                        fmt="*", color=INK, ms=11, capsize=2, label="ResNet-18 full inference")
        ax.set_title(f"{NAMES[cfg]} (mean of {len(seeds)} seeds)")
        ax.set_xlabel("Mean GFLOPs per image (test)")
        ax.set_ylabel("Top-1 accuracy (%)")
        ax.legend(fontsize=7, loc="lower right")
    savefig(fig, out, "tradeoff.png")


def fig_ablation(A, out):
    cfgs = [c for c in ("ee_resnet18_kd", "ee_resnet18_ce", "ee_resnet18_pool") if c in A]
    colors = {"ee_resnet18_kd": BLUE, "ee_resnet18_ce": ORANGE, "ee_resnet18_pool": AQUA}
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.4))
    w = 0.8 / len(cfgs)
    for j, cfg in enumerate(cfgs):
        acc = A[cfg]["mean"]["test_acc"]
        x = np.arange(len(acc)) + (j - (len(cfgs) - 1) / 2) * w
        axes[0].bar(x, [100 * a[0] for a in acc], w * 0.92, yerr=[100 * a[1] for a in acc], capsize=2,
                    color=colors[cfg], label=NAMES[cfg])
    axes[0].set_xticks(range(4), [f"Exit {i + 1}" for i in range(4)])
    axes[0].set_ylim(40, None)
    axes[0].set_ylabel("Top-1 accuracy (%)")
    axes[0].set_title("Accuracy of each exit")
    axes[0].legend(fontsize=7, loc="lower right")
    base = A["resnet18"]["mean"]
    for cfg in cfgs:
        seeds = A[cfg]["seeds"]
        grid = np.linspace(min(x[0] for x in seeds[0]["curves"]["Max-prob"]),
                           max(x[0] for x in seeds[0]["curves"]["Max-prob"]), 160)
        mu, _ = mean_curve([s["curves"]["Max-prob"] for s in seeds], grid)
        axes[1].plot(grid, 100 * mu, color=colors[cfg], label=NAMES[cfg])
    axes[1].plot(base["gflops"][0][0], 100 * base["test_acc"][0][0], "*", color=INK, ms=11, label="ResNet-18 full")
    axes[1].set_xlabel("Mean GFLOPs per image (test)")
    axes[1].set_ylabel("Top-1 accuracy (%)")
    axes[1].set_title("Max-prob exiting")
    axes[1].set_ylim(60, None)
    axes[1].legend(fontsize=7, loc="lower right")
    savefig(fig, out, "ablation_training_heads.png")


def fig_calibration(A, out):
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.2))
    for ax, cfg in zip(axes, ("ee_resnet18_kd", "ee_vit_tiny")):
        if cfg not in A:
            continue
        m = A[cfg]["mean"]
        raw, cal = m["ece_test"]["raw"], m["ece_test"]["calibrated"]
        x = np.arange(len(raw))
        ax.bar(x - 0.18, [100 * v[0] for v in raw], 0.34, yerr=[100 * v[1] for v in raw], capsize=2, color=BLUE,
               label="Raw softmax")
        ax.bar(x + 0.18, [100 * v[0] for v in cal], 0.34, yerr=[100 * v[1] for v in cal], capsize=2, color=ORANGE,
               label="Temperature scaled")
        for i, t in enumerate(m["temperature"]):
            ax.annotate(f"T={t[0]:.2f}", (i, 100 * max(raw[i][0] + raw[i][1], cal[i][0] + cal[i][1])),
                        textcoords="offset points", xytext=(0, 3), ha="center", fontsize=7, color=INK2)
        ax.set_xticks(x, [f"Exit {i + 1}" for i in x])
        ax.set_ylabel("ECE on test (%)")
        ax.set_title(NAMES[cfg])
        ax.legend(fontsize=7.5)
    savefig(fig, out, "calibration_ece.png")


def fig_thresholds(A, out):
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.1))
    for ax, cfg in zip(axes, ("ee_resnet18_kd", "ee_vit_tiny")):
        if cfg not in A:
            continue
        P = A[cfg]["mean"]["policies"]
        names = list(P)
        x = np.arange(len(names))
        g = [P[lab]["test"]["cost"] for lab in names]
        pe = [P[lab]["per_exit"]["test"]["cost"] for lab in names]
        ax.bar(x - 0.18, [v[0] for v in g], 0.34, yerr=[v[1] for v in g], capsize=2, color=BLUE,
               label="One global threshold")
        ax.bar(x + 0.18, [v[0] for v in pe], 0.34, yerr=[v[1] for v in pe], capsize=2, color=ORANGE,
               label="Per-exit thresholds")
        ax.set_xticks(x, names, fontsize=7.5)
        ax.set_ylabel("Mean GFLOPs at selected point")
        ax.set_title(f"{NAMES[cfg]}: cost at the validation-selected point", fontsize=9)
        ax.legend(fontsize=7)
    savefig(fig, out, "threshold_tuning.png")


def _corr_mean(seeds, fn):
    return [np.mean([np.nanmean([fn(s, c, sev) for c in CORRUPTIONS]) for s in seeds]) for sev in SEV_LABELS]


def fig_robustness(A, out):
    lines = [("resnet18", None, INK, "-", "*"), ("ee_resnet18_kd", "Max-prob", BLUE, "-", "o"),
             ("ee_resnet18_kd", "Entropy + TS", YELLOW_SAFE, ":", "o"), ("ee_vit_tiny", "Max-prob", VIOLET, "--", "^")]
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 5.6), sharex=True)
    for ax, c in zip(axes.flat, CORRUPTIONS):
        for cfg, label, color, ls, mk in lines:
            if cfg not in A:
                continue
            seeds = A[cfg]["seeds"]
            if label is None:
                ys = [np.mean([s["corrupt_acc"][c][sev] for s in seeds]) for sev in SEV_LABELS]
                clean = np.mean([s["test_acc"][0] for s in seeds])
                name = NAMES[cfg]
            else:
                ys = [np.mean([s["corrupt"][c][sev][label]["acc"] for s in seeds]) for sev in SEV_LABELS]
                clean = np.mean([s["policies"][label]["test"]["acc"] for s in seeds])
                name = f"{NAMES[cfg]}, {label}"
            ax.plot(range(6), [100 * clean, *[100 * y for y in ys]], marker=mk, color=color, ls=ls, ms=4, label=name)
        ax.set_title(CNAMES[c])
    for ax in axes[1]:
        ax.set_xlabel("Severity (0 = clean)")
    for ax in axes[:, 0]:
        ax.set_ylabel("Top-1 accuracy (%)")
    axes[0, 0].legend(fontsize=6.5)
    savefig(fig, out, "robustness_per_corruption.png")

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
    for cfg, mk in (("ee_resnet18_kd", "o"), ("ee_vit_tiny", "^")):
        if cfg not in A:
            continue
        seeds = A[cfg]["seeds"]
        for label in ("Max-prob", "Entropy + TS"):
            color, ls = POLICY_STYLE[label]
            clean = [np.mean([s["policies"][label]["test"][key] for s in seeds]) for key in ("acc", "cost")]
            acc = _corr_mean(seeds, lambda s, c, v: s["corrupt"][c][v][label]["acc"])
            cost = _corr_mean(seeds, lambda s, c, v: s["corrupt"][c][v][label]["cost"])
            early = _corr_mean(seeds, lambda s, c, v: s["corrupt"][c][v][label]["early_acc"] or np.nan)
            name = f"{NAMES[cfg]}, {label}"
            axes[0].plot(range(6), [100 * clean[0], *[100 * a for a in acc]], marker=mk, color=color, ls=ls, label=name)
            axes[1].plot(range(6), [clean[1], *cost], marker=mk, color=color, ls=ls, label=name)
            axes[2].plot(range(1, 6), [100 * e for e in early], marker=mk, color=color, ls=ls, label=name)
    seeds = A["resnet18"]["seeds"]
    base = [np.mean([s["test_acc"][0] for s in seeds])] + _corr_mean(seeds, lambda s, c, v: s["corrupt_acc"][c][v])
    axes[0].plot(range(6), [100 * b for b in base], "*-", color=INK, ms=7, label="ResNet-18 full")
    axes[1].axhline(A["resnet18"]["mean"]["gflops"][0][0], color=INK, lw=0.9, ls="--")
    axes[1].text(0, A["resnet18"]["mean"]["gflops"][0][0], " ResNet-18 full", va="bottom", fontsize=7, color=INK2)
    axes[0].set(xlabel="Severity (0 = clean)", ylabel="Top-1 accuracy (%)", title="Accuracy, mean over 6 corruptions")
    axes[1].set(xlabel="Severity (0 = clean)", ylabel="Mean GFLOPs per image", title="Compute spent")
    axes[2].set(xlabel="Severity", ylabel="Accuracy of early-exited images (%)", title="Reliability of early exits")
    axes[0].legend(fontsize=6.5)
    savefig(fig, out, "robustness_summary.png")


def fig_exit_shift(A, out):
    if "ee_resnet18_kd" not in A:
        return
    seeds = A["ee_resnet18_kd"]["seeds"]
    rows = []
    for label in ("Max-prob", "Entropy + TS"):
        rows.append((f"{label}: clean", np.mean([s["policies"][label]["test"]["exit_frac"] for s in seeds], 0)))
        for sev in ("1", "3", "5"):
            frac = np.mean([[s["corrupt"][c][sev][label]["exit_frac"] for c in CORRUPTIONS] for s in seeds], (0, 1))
            rows.append((f"{label}: severity {sev}", frac))
    fig, ax = plt.subplots(figsize=(7.0, 0.36 * len(rows) + 1.0))
    for j, (lab, frac) in enumerate(rows):
        left = 0
        for i, f in enumerate(frac):
            ax.barh(j, 100 * f, left=left, color=EXIT_RAMP[i], edgecolor="white", linewidth=1,
                    label=f"Exit {i + 1}" if j == 0 else None)
            if f > 0.06:
                ax.text(left + 50 * f, j, f"{100 * f:.0f}", ha="center", va="center", fontsize=7,
                        color="white" if i > 0 else INK)
            left += 100 * f
    ax.set_yticks(range(len(rows)), [r[0] for r in rows], fontsize=7.5)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Share of test images (%), EE-ResNet-18, mean over corruptions and seeds")
    ax.grid(axis="y", visible=False)
    ax.legend(ncol=4, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, 1.14))
    savefig(fig, out, "exit_shift.png")


def fig_corruption_ece(A, out):
    if "ee_resnet18_kd" not in A:
        return
    seeds = A["ee_resnet18_kd"]["seeds"]
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.0), sharey=True)
    for ax, key, title in ((axes[0], "raw", "Raw softmax"), (axes[1], "calibrated", "Temperature scaled (clean val)")):
        for i in range(4):
            clean = np.mean([s["ece_test"][key][i] for s in seeds])
            ys = _corr_mean(seeds, lambda s, c, v: s["corrupt_ece"][c][v][key][i])
            ax.plot(range(6), [100 * clean, *[100 * y for y in ys]], "o-", color=EXIT_RAMP[i], label=f"Exit {i + 1}")
        ax.set_title(title)
        ax.set_xlabel("Severity (0 = clean)")
    axes[0].set_ylabel("ECE (%), mean over corruptions")
    axes[0].legend(fontsize=7.5)
    savefig(fig, out, "corruption_ece.png")


def latency_rows(A):
    """Measured latency of each seed-0 model at its validation-selected max-prob point."""
    rows = []
    for cfg in ("resnet18", "ee_resnet18_kd", "ee_resnet18_pool", "ee_resnet18_ce", "ee_vit_tiny"):
        lat = next((s.get("latency") for s in A.get(cfg, {}).get("raw", []) if s["seed"] == 0), None)
        if not lat:
            continue
        for dev in ("gpu", "cpu1", "cpu4"):
            if dev not in lat:
                continue
            row = {"cfg": cfg, "device": dev, "device_name": lat[dev]["device"], "full_ms": lat[dev]["forced_ms"][-1],
                   "forced_ms": lat[dev]["forced_ms"]}
            for a in lat[dev]["adaptive"]:
                key = {"strict": "ops", "relaxed": "ops_relaxed"}.get(a.get("point"))
                if key:
                    row.setdefault(key, {})[a["label"]] = a
            rows.append(row)
        if "gpu" in lat and "batched" in lat["gpu"]:
            b = lat["gpu"]["batched"]
            row = {"cfg": cfg, "device": "gpu_batched", "full_tp": b["full"]}
            for a in b["adaptive"]:
                key = {"strict": "ops", "relaxed": "ops_relaxed"}.get(a.get("point"))
                if key:
                    row.setdefault(key, {})[a["label"]] = a
            rows.append(row)
    return rows


def fig_latency(rows, out):
    if not rows:
        return
    items = [("resnet18", None, "ResNet-18, full", INK2), ("ee_resnet18_kd", None, "EE-ResNet-18, full path", "#86b6ef"),
             ("ee_resnet18_kd", "Max-prob", "EE-ResNet-18, max-prob", BLUE),
             ("ee_resnet18_pool", "Max-prob", "EE-ResNet-18 pooling heads, max-prob", AQUA),
             ("ee_vit_tiny", None, "EE-ViT-Tiny, full path", "#9085e9"),
             ("ee_vit_tiny", "Max-prob", "EE-ViT-Tiny, max-prob", VIOLET)]
    devs = [("gpu", "GPU (T4), batch 1"), ("cpu1", "CPU, 1 thread, batch 1"), ("cpu4", "CPU, 4 threads, batch 1")]
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.0), sharey=True)
    for ax, (dev, title) in zip(axes, devs):
        rs = {r["cfg"]: r for r in rows if r["device"] == dev}
        if "resnet18" not in rs:
            continue
        base = rs["resnet18"]["full_ms"]
        bars = []
        for cfg, label, name, color in items:
            if cfg in rs:
                v = rs[cfg]["full_ms"] if label is None else rs[cfg].get("ops", {}).get(label, {}).get("ms")
                if v:
                    bars.append((name, v, color))
        y = np.arange(len(bars))
        ax.barh(y, [b[1] for b in bars], 0.7, color=[b[2] for b in bars])
        for i, b in enumerate(bars):
            ax.text(b[1], i, f" {b[1]:.1f} ms, {base / b[1]:.2f}x", va="center", fontsize=6.5, color=INK2)
        ax.set_yticks(y, [b[0] for b in bars], fontsize=7)
        ax.invert_yaxis()
        ax.set_xlim(0, max(b[1] for b in bars) * 1.45)
        ax.set_xlabel("ms per image")
        ax.set_title(title, fontsize=9)
        ax.grid(axis="y", visible=False)
    savefig(fig, out, "latency.png")


def fig_training(A, out):
    cfgs = [c for c in NAMES if c in A]
    fig, axes = plt.subplots(1, len(cfgs), figsize=(2.6 * len(cfgs), 2.6), sharey=True)
    for ax, cfg in zip(np.atleast_1d(axes), cfgs):
        h = next(s for s in A[cfg]["raw"] if s["seed"] == 0)["history"]
        k = len(h[0]["val_acc"])
        for i in range(k):
            ax.plot([x["epoch"] for x in h], [100 * x["val_acc"][i] for x in h], color=INK if k == 1 else EXIT_RAMP[i],
                    lw=1.2, label="Final exit" if k == 1 else f"Exit {i + 1}")
        ax.set_title(NAMES[cfg].replace(", ", ",\n"), fontsize=8.5)
        ax.set_xlabel("Epoch")
    np.atleast_1d(axes)[0].set_ylabel("Validation accuracy (%)")
    np.atleast_1d(axes)[-1].legend(fontsize=6.5, loc="lower right")
    savefig(fig, out, "training_curves.png")


def analyze_final_main():
    p = argparse.ArgumentParser()
    p.add_argument("--runs", default="runs")
    p.add_argument("--out", default="results/final")
    args = p.parse_args()
    os.makedirs(os.path.join(args.out, "figures"), exist_ok=True)
    runs = discover(args.runs)
    print({k: [r["seed"] for r in v] for k, v in runs.items()}, flush=True)
    base_cost = runs["resnet18"][0]["flops"]["cumulative"][0] / 1e9
    base_acc = float(np.mean([r["acc"]["test"][0] for r in runs["resnet18"]]))
    A = {}
    for cfg, rs in runs.items():
        vit = cfg == "ee_vit_tiny"
        ref = rs[0]["flops"]["cumulative"][-1] / 1e9 if vit else base_cost
        seeds = [analyze_run(r, ref, None if vit else base_acc) for r in rs]
        A[cfg] = {"raw": rs, "seeds": seeds, "mean": aggregate(seeds), "n": len(seeds)}
        print("analyzed", cfg, flush=True)
    lat = latency_rows(A)
    for name in ("dataset_samples.png", "corruption_examples.png"):
        src = os.path.join("results", "interim", "figures", name)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(args.out, "figures", name))
    fig_training(A, args.out)
    fig_tradeoff(A, args.out)
    fig_ablation(A, args.out)
    fig_calibration(A, args.out)
    fig_thresholds(A, args.out)
    fig_robustness(A, args.out)
    fig_exit_shift(A, args.out)
    fig_corruption_ece(A, args.out)
    fig_latency(lat, args.out)
    summary = {cfg: {"n": v["n"], "mean": {k: x for k, x in v["mean"].items() if k != "curves"}}
               for cfg, v in A.items()}
    summary["latency"] = lat
    with open(os.path.join(args.out, "summary.json"), "w") as f:
        json.dump(summary, f, indent=1, default=float)
    print("wrote", os.path.join(args.out, "summary.json"))


In [ ]:
cli(analyze_final_main, "--runs", RUNS_DIR, "--out", RESULTS_DIR)
S = json.load(open(os.path.join(RESULTS_DIR, "summary.json")))
b = S["resnet18"]["mean"]
print(f"ResNet-18: {100 * b['test_acc'][0][0]:.2f}% test accuracy, {b['gflops'][0][0]:.3f} GFLOPs")
for cfg in ("ee_resnet18_kd", "ee_vit_tiny"):
    if cfg not in S: continue
    m = S[cfg]["mean"]
    ref = b["gflops"][0][0] if cfg != "ee_vit_tiny" else m["gflops"][-1][0]
    print(f"\n{cfg}: per-exit accuracy", [round(100 * a[0], 2) for a in m["test_acc"]])
    for label, p in m["policies"].items():
        t = p["test"]
        print(f"  {label:14s} {100 * t['acc'][0]:6.2f}%  {t['cost'][0]:.3f} GFLOPs ({100 * (1 - t['cost'][0] / ref):.0f}% saved)"
              f"  exits {[round(100 * f[0]) for f in t['exit_frac']]}")
for fig_name in ("training_curves", "tradeoff", "calibration_ece", "threshold_tuning", "latency",
                 "robustness_summary", "exit_shift", "corruption_ece"):
    path = os.path.join(RESULTS_DIR, "figures", fig_name + ".png")
    if os.path.exists(path):
        display(Markdown(f"**{fig_name.replace('_', ' ')}**")); display(Image(path))

## 9. Our full-scale results

The cells above reproduce the pipeline on one seed. Our reported numbers come from 200-epoch runs with three seeds
on Kaggle T4 GPUs, two ablations, and a follow-up study of 20 more runs. Their summaries and figures are in the
repository, loaded below.

In [ ]:
REPO_DIR = os.path.join(WORK, "repo")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/goyaladitya05/adaptive-dnn-inference", REPO_DIR], check=True)
FINAL = json.load(open(os.path.join(REPO_DIR, "results/final/summary.json")))
import pandas as pd
rows = []
names = {"resnet18": "ResNet-18", "ee_resnet18_kd": "EE-ResNet-18", "ee_resnet18_ce": "EE-ResNet-18, no distillation",
         "ee_resnet18_pool": "EE-ResNet-18, pooling heads", "ee_vit_tiny": "EE-ViT-Tiny"}
for cfg, label in names.items():
    m = FINAL[cfg]["mean"]
    accs = [f"{100 * a[0]:.2f} ± {100 * a[1]:.2f}" for a in m["test_acc"]]
    rows.append([label, FINAL[cfg]["n"], *(["-"] * (4 - len(accs)) + accs)])
display(pd.DataFrame(rows, columns=["Model", "Seeds", "Exit 1", "Exit 2", "Exit 3", "Final exit"]).set_index("Model"))

ref = FINAL["resnet18"]["mean"]["gflops"][0][0]
rows = []
for label, p in FINAL["ee_resnet18_kd"]["mean"]["policies"].items():
    for kind, t in (("global", p["test"]), ("per-exit", p["per_exit"]["test"])):
        rows.append([label, kind, f"{100 * t['acc'][0]:.2f}", f"{t['cost'][0]:.3f}", f"{100 * (1 - t['cost'][0] / ref):.0f}%"])
display(Markdown("**EE-ResNet-18 operating points (no accuracy loss on validation), mean of 3 seeds**"))
display(pd.DataFrame(rows, columns=["Exit rule", "Thresholds", "Test acc. (%)", "GFLOPs", "Saved"]))
for fig_name in ("tradeoff", "ablation_training_heads", "calibration_ece", "latency", "robustness_summary", "exit_shift"):
    display(Markdown(f"**{fig_name.replace('_', ' ')}**"))
    display(Image(os.path.join(REPO_DIR, "results/final/figures", fig_name + ".png")))

### Follow-up study: what makes early exits reliable under corruption?

We crossed architecture (CNN, ViT) with training recipe (plain, strong = label smoothing + mixup/CutMix +
RandAugment), varied the distillation weight (0 to 0.75) and ablated the recipe components, with three seeds per
main configuration. The decisive comparison is at **matched clean computation**: at the same clean GFLOPs, how
many severely corrupted images exit early, and how many of those early exits are correct.

In [ ]:
MC = json.load(open(os.path.join(REPO_DIR, "results/study/matched_cost.json")))
rows = []
for cfg, v in MC.items():
    budget = "0.6" if "resnet" in cfg else "0.4"
    for pol, label in (("raw", "raw"), ("ts", "TS (clean)"), ("loco", "TS (other corruptions)")):
        if pol in v["5"]:
            r = v["5"][pol][budget]
            rows.append([cfg.replace("ee_", ""), label, budget, f"{100 * r['early_frac'][0]:.1f}", f"{100 * r['early_acc'][0]:.1f}"])
display(pd.DataFrame(rows, columns=["Configuration", "Calibration", "Clean GFLOPs", "Early exits at sev. 5 (%)",
                                    "Early-exit accuracy (%)"]))
for fig_name in ("lead2_architecture_recipe", "lead23_temperature"):
    display(Image(os.path.join(REPO_DIR, "results/study/figures", fig_name + ".png")))

## 10. Conclusions

* **Early exits save computation.** With thresholds chosen so that no accuracy is lost on validation, EE-ResNet-18
  needs about half the FLOPs of ResNet-18 (51% fewer, 55% with per-exit thresholds), and 65% of test images stop
  at the first exit. Its final exit matches ResNet-18 (76.1%).
* **FLOPs are not latency.** The saving is a 1.9x speed-up on one CPU thread, but only 1.06x on a T4 GPU at
  batch 1, where per-exit synchronisation dominates; batching with exited images removed recovers 1.37x.
* **Calibration.** CNN exits are overconfident and ViT exits (trained with label smoothing and mixup)
  underconfident; temperature scaling brings every exit to at most about 4% ECE, but barely changes how exits rank
  images, so per-exit thresholds are the better tool for efficiency.
* **Under corruption**, confident exits keep stopping images they misclassify (Mehra et al., 2022, observed the
  same for multi-exit CNNs). At matched clean computation, calibration (temperature scaling, distillation strength,
  label smoothing, mixup) moves a model only a few points along its cost and reliability curve, while
  augmentation-based robustness (RandAugment, the strong recipe) roughly doubles the accuracy of early exits under
  severe corruption: **calibration moves a model along the curve; robustness moves the curve.**